# 00 · 환경 점검

4-2 는 4-1 과 **같은 도커 환경 · 같은 DB** 를 쓴다. 새로 설치할 것은 없다.

| 부 | 노트북 | 데이터 |
|---|---|---|
| 1부 원리 | 10 NSW · 11 HNSW · 12 IVF · 13 PQ | numpy 로 만든 점 (DB 안 씀) |
| 2부 pgvector | 20 데이터·정답지 · 21 HNSW · 22 IVFFlat · 23 양자화 | 합성 벡터 10만 개 (20번이 만듦) |
| 3부 품질 | 30 퓨전 · 31 분포 진단 | KLUE-MRC · Qwen3 임베딩 (4-1 실습1 결과) |

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 버전과 인덱스 빌드 설정 |
| 3 | 3부 준비 확인 — 4-1 실습1 의 결과가 있나 |

**Run → Run All Cells**

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"

### 셀 2 · 버전과 인덱스 빌드 설정
- pgvector 0.8 이상 : `hnsw.iterative_scan` 등 최신 기능
- maintenance_work_mem 1GB · 병렬 빌드 워커 4 : 21·22번 인덱스 빌드 속도 (4-1 docker-compose.yml 의 db 설정)

In [ ]:
# 셀 2 · 버전과 인덱스 빌드 설정
from labutil import *

conn = connect(DB_URL)
q(conn, """
SELECT split_part(version(), ' ', 2)                                          AS postgres,
       (SELECT extversion FROM pg_extension WHERE extname = 'vector')        AS pgvector,
       (SELECT extversion FROM pg_extension WHERE extname = 'textsearch_ko') AS textsearch_ko,
       current_setting('maintenance_work_mem')                                AS maintenance_work_mem,
       current_setting('max_parallel_maintenance_workers')                    AS 병렬빌드_워커
""").T

### 셀 3 · 3부 준비 확인
30·31번은 4-1 실습1 이 채운 지문 · 질문 임베딩과 BM25 역색인을 그대로 쓴다. 모두 ✔ 이면 준비 완료.
1부·2부는 이 결과와 상관없이 바로 할 수 있다.

In [ ]:
# 셀 3 · 3부 준비 확인
checks = q(conn, """
SELECT '지문 임베딩' AS 항목, count(embedding) || ' / ' || count(*) AS 값, count(embedding) = count(*) AND count(*) > 0 AS ok FROM docs
UNION ALL
SELECT '질문 임베딩', count(embedding) || ' / ' || count(*), count(embedding) = count(*) AND count(*) > 0 FROM questions
UNION ALL
SELECT 'BM25 역색인 행', count(*)::text, count(*) > 0 FROM bm25_tf
UNION ALL
SELECT 'emb128 HNSW 인덱스', coalesce(to_regclass('docs_emb128_hnsw')::text, '없음'), to_regclass('docs_emb128_hnsw') IS NOT NULL
""")
checks["ok"] = checks["ok"].map({True: "✔", False: "✘ 4-1 실습1 을 끝까지"})
checks